## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is usefull for ensuring the outputs can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output

### Pydantic

pydantic models provides the richest feature set with field validation, desciptions and nested structures.

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

model = init_chat_model("groq:qwen/qwen3.8-27b")

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This is the year the movie was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The rating of the movie out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'Qwen3.8 27B', 'status': 'beta', 'release_date': '2026-08-14', 'last_updated': '2026-08-14', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x112a44440>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x112a44ec0>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title'

In [5]:
response = model_with_structure.invoke("PRovide details about the movie Inception by Chris Nolan")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message Output alongside parsed structure 

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="This is the year the movie was released")
    director: str = Field(...,description="The director of the movie")
    rating: float = Field(...,description="The rating of the movie out of 10")


model_with_structure = model.with_structured_output(Movie, include_raw = True)

response = model_with_structure.invoke(
    "Provide details about the movie Shrek")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '185tqvxx3', 'function': {'arguments': '{"director":"Andrew Adamson, Vicky Jenson","rating":8.3,"title":"Shrek","year":2001}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 353, 'total_tokens': 427, 'completion_time': 0.19822726, 'completion_tokens_details': None, 'prompt_time': 0.025429368, 'prompt_tokens_details': None, 'queue_time': 0.074769119, 'total_time': 0.223656628}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a20c15-2bee-702c-91d3-646401b27278-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Andrew Adamson, Vicky Jenson', 'rating': 8.3, 'title': 'Shrek', 'year': 2001}, 'id': '185tqvxx3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 353, 'output_tokens

### Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie The Wolf Of Wall Street")
response

MovieDetails(title='The Wolf of Wall Street', year=2013, cast=[Actor(name='Leonardo DiCaprio', role='Jordan Belfort'), Actor(name='Jonah Hill', role='Donnie Azoff'), Actor(name='Margot Robbie', role='Naomi Lapaglia'), Actor(name='Matthew McConaughey', role='Mark Hunter'), Actor(name='Kyle Chandler', role='Dickie J. Bonner'), Actor(name='Rob Reiner', role='C. Michael (Broker)')], genres=['Biography', 'Comedy', 'Crime', 'Drama'], budget=100.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [11]:
from typing_extensions import TypedDict, Annotated

class movieDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of the movie release"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]


model_with_typeddict = model.with_structured_output(movieDict)
response = model_with_typeddict.invoke("Provide details of the movie avengers age of ultron")
response

{'director': 'Joss Whedon',
 'rating': 7.3,
 'title': 'Avengers: Age of Ultron',
 'year': 2015}

In [14]:
class Actor(TypedDict):
    name: str
    role: str


class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie The Wolf Of Wall Street")
response

{'budget': 100000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Jordan Belfort'},
  {'name': 'Margot Robbie', 'role': 'Naomi Lapaglia'},
  {'name': 'Jonah Hill', 'role': 'Donnie Azoff'},
  {'name': 'Matthew McConaughey', 'role': 'Mark Hanna'},
  {'name': 'Kyle Chandler', 'role': 'FBI Agent Robert Levinson'},
  {'name': 'Rob Reiner', 'role': 'Jean-Baptiste'},
  {'name': 'Jon Favreau', 'role': 'Sam Rabkin'},
  {'name': 'Jean Dujardin', 'role': 'Bradka'},
  {'name': 'Jack Huston', 'role': 'Danny Porush'},
  {'name': "Denis O'Hare", 'role': 'Curtis Kull'}],
 'genres': ['Biography', 'Crime', 'Drama', 'Comedy'],
 'title': 'The Wolf of Wall Street',
 'year': 2013}

In [15]:
model.profile

{'name': 'Qwen3.8 27B',
 'status': 'beta',
 'release_date': '2026-08-14',
 'last_updated': '2026-08-14',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True}

### DataClasses

A DataClass is a class typically containing mainly data, although there aren't really any restrictions. We create it using the @dataclass decorator

In [ ]:
## Using Pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = "groq:qwen/qwen3.8-27b",
    response_format = ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, Male , john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [19]:
## Using TypedDict

from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [20]:
## DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')